# 1.3 · 版本、特性生命周期与 2026-07-28 的变化

**Versioning, feature lifecycle and what changed in 2026-07-28**

> **学完本课你能：** 逐请求协商协议版本，解释双时代客户端与服务器如何与 2025 时代的 MCP 互操作，并列出 2025-11-25 到 2026-07-28 之间的每一项重大变更与弃用。

官方阅读材料：[Versioning](https://modelcontextprotocol.io/docs/2026-07-28/learn/versioning) · [Spec: Versioning and compatibility](https://modelcontextprotocol.io/specification/2026-07-28/basic/versioning) · [Key changes (changelog)](https://modelcontextprotocol.io/specification/2026-07-28/changelog) · [Deprecated features](https://modelcontextprotocol.io/specification/2026-07-28/deprecated) · [Feature lifecycle policy](https://modelcontextprotocol.io/community/feature-lifecycle)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 版本字符串和一个协商函数
版本是一个枚举集合。客户端和服务器各自支持其中的某个子集；客户端想要的是双方共有的最新版本。顺序来自一个显式的注册表，而不是字符串比较。这就是版本协商（Version Negotiation）的核心。

In [ ]:
import json

KNOWN_VERSIONS = ["2024-11-05", "2025-03-26", "2025-06-18", "2025-11-25", "2026-07-28"]  # oldest -> newest
MODERN = {"2026-07-28"}   # per-request _meta era; everything older uses the initialize handshake


def choose_version(client_supports, server_supports, known=KNOWN_VERSIONS):
    """Newest version both sides support, or None. Unknown strings are ignored, never 'newer'."""
    mutual = [v for v in known if v in client_supports and v in server_supports]
    return mutual[-1] if mutual else None


cases = {
    "both modern":               (["2026-07-28", "2025-11-25"], ["2026-07-28"]),
    "server still on 2025":      (["2026-07-28", "2025-11-25"], ["2025-06-18", "2025-11-25"]),
    "no overlap":                (["2026-07-28"], ["2025-03-26"]),
    "server claims 'zzz'":       (["2026-07-28"], ["zzz", "2026-07-28"]),
}
for label, (client, server) in cases.items():
    picked = choose_version(client, server)
    era = "-" if picked is None else ("modern" if picked in MODERN else "legacy handshake")
    print(f"{label:<22} -> {picked}  ({era})")

### 2. 本笔记本后续要用到的三个服务器
我们需要每个时代各一个服务器。SDK 的 `MCPServer` 是<strong>双时代（Dual-era）</strong>的。另外两个是很小的手写 JSON-RPC 循环：一个只懂 `initialize` 的<strong>旧版</strong>服务器（就像任何一个还没升级的 2025 服务器），以及一个<strong>只支持新版</strong>的服务器，它会按照规范的建议，用写明其版本的 `-32022` 拒绝 `initialize`。一个小小的<strong>线路监听（wiretap）</strong>脚本位于客户端和服务器之间，记录双向的每一行。

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())

(workdir / "dual_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("weather", version="2.3.0")

    @server.tool()
    def weather_current(location: str) -> str:
        """Get current weather for a location."""
        return f"{location}: 20 degrees (dual-era SDK server)"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

(workdir / "legacy_server.py").write_text(textwrap.dedent('''
    """A 2025-11-25-era server: initialize handshake or nothing."""
    import json, sys
    initialized = False
    def reply(id_, result=None, error=None):
        print(json.dumps({"jsonrpc": "2.0", "id": id_, **({"error": error} if error else {"result": result})}), flush=True)
    for line in sys.stdin:
        msg = json.loads(line); method, id_ = msg.get("method"), msg.get("id")
        if method == "initialize":
            reply(id_, {"protocolVersion": "2025-11-25", "capabilities": {"tools": {}},
                        "serverInfo": {"name": "old-weather", "version": "0.9.0"}})
        elif method == "notifications/initialized":
            initialized = True
        elif not initialized:
            reply(id_, error={"code": -32600, "message": "Server not initialized"})
        elif method == "tools/list":
            reply(id_, {"tools": [{"name": "weather_current", "description": "Current weather",
                                   "inputSchema": {"type": "object", "properties": {"location": {"type": "string"}}}}]})
        elif method == "tools/call":
            reply(id_, {"content": [{"type": "text", "text": "Sunny (legacy 2025 server)"}]})
        elif method == "ping":
            reply(id_, {})
        elif id_ is not None:
            reply(id_, error={"code": -32601, "message": f"Method not found: {method}"})
'''))

(workdir / "modern_only_server.py").write_text(textwrap.dedent('''
    """A server that speaks 2026-07-28 and nothing else."""
    import json, sys
    SUPPORTED = ["2026-07-28"]
    INFO = {"io.modelcontextprotocol/serverInfo": {"name": "new-weather", "version": "3.0.0"}}
    def send(msg): print(json.dumps({"jsonrpc": "2.0", **msg}), flush=True)
    for line in sys.stdin:
        msg = json.loads(line); method, id_ = msg.get("method"), msg.get("id")
        if id_ is None:
            continue
        if method == "initialize":   # spec: name supported versions; legacy clients cannot fall forward
            send({"id": id_, "error": {"code": -32022, "message": "This server only speaks 2026-07-28",
                  "data": {"supported": SUPPORTED, "requested": msg["params"].get("protocolVersion")}}})
            continue
        version = msg.get("params", {}).get("_meta", {}).get("io.modelcontextprotocol/protocolVersion")
        if version is None:
            send({"id": id_, "error": {"code": -32602, "message": "missing _meta protocolVersion"}})
        elif version not in SUPPORTED:
            send({"id": id_, "error": {"code": -32022, "message": "Unsupported protocol version",
                  "data": {"supported": SUPPORTED, "requested": version}}})
        elif method == "server/discover":
            send({"id": id_, "result": {"resultType": "complete", "supportedVersions": SUPPORTED,
                  "capabilities": {"tools": {}}, "ttlMs": 60000, "cacheScope": "public", "_meta": INFO}})
        elif method == "tools/list":
            send({"id": id_, "result": {"resultType": "complete", "ttlMs": 60000, "cacheScope": "public", "_meta": INFO,
                  "tools": [{"name": "weather_current", "description": "Current weather",
                             "inputSchema": {"type": "object", "properties": {"location": {"type": "string"}}}}]}})
        elif method == "tools/call":
            send({"id": id_, "result": {"resultType": "complete", "_meta": INFO,
                  "content": [{"type": "text", "text": "Sunny (modern-only server)"}]}})
        else:
            send({"id": id_, "error": {"code": -32601, "message": f"Method not found: {method}"}})
'''))

(workdir / "wiretap.py").write_text(textwrap.dedent('''
    """Usage: python wiretap.py LOGFILE server.py -- relays stdio and logs each line."""
    import subprocess, sys, threading
    log_path, target = sys.argv[1], sys.argv[2:]
    child = subprocess.Popen([sys.executable, *target], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
    lock = threading.Lock()
    def record(prefix, line):
        with lock, open(log_path, "a") as f:
            f.write(prefix + " " + line.rstrip("\\n") + "\\n")
    def pump():
        for line in child.stdout:
            record("<<<", line); sys.stdout.write(line); sys.stdout.flush()
    t = threading.Thread(target=pump, daemon=True); t.start()
    for line in sys.stdin:
        record(">>>", line); child.stdin.write(line); child.stdin.flush()
    child.stdin.close(); child.wait(); t.join(timeout=2)
'''))

SERVERS = {"dual-era": workdir / "dual_server.py",
           "legacy": workdir / "legacy_server.py",
           "modern-only": workdir / "modern_only_server.py"}
print("written:", sorted(p.name for p in workdir.iterdir()))

### 3. 手动进行逐请求协商：`-32022` 与重试
一个未来的客户端偏好一个（假想的）更新修订版 `2027-03-01`，同时也支持 `2026-07-28`。它不做任何发现就直接发送请求，从错误中读取 `data.supported`，进行协商，然后重试——这正是版本规范中描述的流程。

In [ ]:
CLIENT_SUPPORTS = ["2027-03-01", "2026-07-28"]   # newest first; 2027-03-01 is made up for the demo


def envelope(version):
    return {"io.modelcontextprotocol/protocolVersion": version,
            "io.modelcontextprotocol/clientCapabilities": {},
            "io.modelcontextprotocol/clientInfo": {"name": "future-client", "version": "1.0"}}


proc = subprocess.Popen([sys.executable, str(SERVERS["dual-era"])],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)


def rpc(message):
    print(">>>", json.dumps(message)[:200])
    proc.stdin.write(json.dumps(message) + "\n"); proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    print("<<<", json.dumps(reply)[:300], "\n")
    return reply


def call_with_negotiation(request_id, method, params):
    version = CLIENT_SUPPORTS[0]
    for attempt in range(2):
        reply = rpc({"jsonrpc": "2.0", "id": request_id + attempt, "method": method,
                     "params": {**params, "_meta": envelope(version)}})
        error = reply.get("error")
        if not error or error["code"] != -32022:
            return version, reply
        mutual = [v for v in CLIENT_SUPPORTS if v in error["data"]["supported"]]
        if not mutual:
            raise RuntimeError(f"no common version: server supports {error['data']['supported']}")
        version = mutual[0]
        print(f"... server rejected {error['data']['requested']}, retrying with {version}\n")
    raise RuntimeError("negotiation did not converge")


version, reply = call_with_negotiation(1, "tools/call", {"name": "weather_current", "arguments": {"location": "Oslo"}})
print("negotiated:", version, "| result:", reply["result"]["content"][0]["text"])
proc.stdin.close()
exit_code = proc.wait(timeout=10)

注意这里没有握手：<em>同一个</em> `tools/call` 只是换了一个 `_meta` 版本和一个新的 `id`，被重新发送了一次。

### 4. SDK 的两种连接模式在线路上的样子
`Client(..., mode="legacy")` 强制使用 2025 的 `initialize` 握手；默认的 `mode="auto"` 则用 `server/discover` 进行探测，只有当回复不能证明服务器是新版时才回退。同一个双时代（Dual-era）服务器，两段截然不同的对话。

In [ ]:
import os

from mcp import Client, StdioServerParameters


def tapped(server_key, log_name):
    log = workdir / log_name
    log.unlink(missing_ok=True)
    return log, StdioServerParameters(command=sys.executable, args=[str(workdir / "wiretap.py"), str(log), str(SERVERS[server_key])])


def print_wire(log, width=260):
    for line in log.read_text().splitlines():
        print(line[:width])


for mode in ["legacy", "auto"]:
    log, params = tapped("dual-era", f"wire_{mode}.log")
    async with Client(params, mode=mode) as client:
        result = await client.call_tool("weather_current", {"location": "Paris"})
        print(f"===== mode={mode!r}: negotiated {client.protocol_version}, "
              f"tools.listChanged={client.server_capabilities.tools.list_changed}")
    print_wire(log)
    print()

把两份日志并排读一读：<strong>legacy</strong> 先发送 `initialize`，然后是 `notifications/initialized`，然后是不带 `_meta` 的裸请求，结果中没有 `resultType`。<strong>auto</strong> 先发送 `server/discover`，之后的每个请求都携带完整的信封；结果中带有 `resultType`、`ttlMs`/`cacheScope` 和 `serverInfo`。（SDK 还会在 `tools/call` 之后发出一个 `tools/list`，用来校验工具的输出模式（Output Schema）——阅读日志时知道这一点会很有用。）

### 5. 自动模式对上旧版服务器：探测与回退
现在把 `mode="auto"` 指向旧版服务器。探测得到了一个非新版错误（`-32600 Server not initialized`），于是客户端回退到 `initialize`——事先没有被告知任何关于这个服务器的信息。旧版结果中没有 `resultType`；客户端按规范要求把它视为 `"complete"`。这就是时代探测（Era Probe）的完整过程。

In [ ]:
log, params = tapped("legacy", "wire_auto_vs_legacy.log")
async with Client(params) as client:   # mode="auto" is the default
    result = await client.call_tool("weather_current", {"location": "Paris"})
    print("negotiated:", client.protocol_version, "| server:", client.server_info.name)
    print("result:", result.content[0].text, "| resultType seen by the app:", result.result_type)
print()
print_wire(log)

### 6. 实时运行的兼容性矩阵
客户端模式与服务器时代的每一种组合。`mode="2026-07-28"` 固定使用一个新版版本而不进行探测——即 SDK 中的“只支持新版的客户端”。

In [ ]:
async def attempt(mode, server_key):
    params = StdioServerParameters(command=sys.executable, args=[str(SERVERS[server_key])])
    try:
        async with Client(params, mode=mode, read_timeout_seconds=5) as client:
            result = await client.call_tool("weather_current", {"location": "Paris"})
            return f"works ({client.protocol_version})"
    except Exception as exc:
        while isinstance(exc, BaseExceptionGroup):   # unwrap anyio task-group wrappers
            exc = exc.exceptions[0]
        return f"FAILS: {str(exc)[:55]}"


modes = {"legacy": "legacy client", "auto": "dual-era client", "2026-07-28": "modern-only client"}
for mode, label in modes.items():
    for server_key in SERVERS:
        outcome = await attempt(mode, server_key)
        print(f"{label:<19} x {server_key + ' server':<19} -> {outcome}")
    print()

与规范中的矩阵对比一下：旧版 × 只支持新版会失败（而我们的服务器在错误中写明了支持的版本——这是旧版客户端唯一能展示的诊断信息）；只支持新版 × 旧版会失败；双时代（Dual-era）客户端在所有情况下都能工作。

### 7. SDK 中的弃用与移除
Tier 1 SDK 必须标记已弃用或已移除的 API，并且应当在运行时发出警告。`ping` 在 2026-07-28 中被<strong>移除</strong>：SDK 仍然为旧版连接提供 `send_ping`，会用一条警告标记它，而新版服务器则回答 `-32601 Method not found`。这体现了特性生命周期（Feature Lifecycle）在 SDK 中的落地。

In [ ]:
import warnings

for mode in ["legacy", "auto"]:
    params = StdioServerParameters(command=sys.executable, args=[str(SERVERS["dual-era"])])
    async with Client(params, mode=mode) as client:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            try:
                await client.send_ping()
                outcome = "pong (empty result)"
            except Exception as exc:
                outcome = f"error: {exc}"
        print(f"mode={mode:<7} protocol={client.protocol_version}  ping -> {outcome}")
        for w in caught:
            print(f"   {w.category.__name__}: {w.message}")

### 8. 针对 2025 时代流量的迁移检查器
工程师们会继承旧代码和旧日志。把变更日志和弃用注册表编码成数据，然后扫描第 4 节捕获的旧版线路日志（再加上几段额外的片段），找出任何在 2026-07-28 中被移除或弃用的内容。

In [ ]:
RULES = [  # (needle, state, what to do instead)
    ('"method":"initialize"', "removed", "per-request _meta envelope; optional server/discover"),
    ("notifications/initialized", "removed", "nothing: there is no handshake"),
    ('"method":"ping"', "removed", "no protocol replacement (removed, not deprecated)"),
    ("logging/setLevel", "removed", "io.modelcontextprotocol/logLevel in each request's _meta"),
    ("resources/subscribe", "removed", "subscriptions/listen with resourceSubscriptions"),
    ("Mcp-Session-Id", "removed", "no sessions: server-minted handles passed as tool arguments"),
    ("Last-Event-ID", "removed", "re-issue the request with a new request ID"),
    ("notifications/roots/list_changed", "removed", "pass paths via tool parameters / config"),
    ("tasks/result", "removed", "Tasks extension: poll tasks/get"),
    ("notifications/elicitation/complete", "removed", "client retries the original request (MRTR)"),
    ("sampling/createMessage", "deprecated", "call an LLM provider API directly (SEP-2577)"),
    ("roots/list", "deprecated", "pass directories via tool parameters, resource URIs or config"),
    ("notifications/message", "deprecated", "log to stderr or use OpenTelemetry"),
    ('"includeContext":"thisServer"', "deprecated", 'omit includeContext or use "none"'),
    ("registration_endpoint", "deprecated", "Client ID Metadata Documents instead of DCR"),
]


def lint(lines):
    findings = {}
    for line in lines:
        compact = line.replace(" ", "")
        for needle, state, fix in RULES:
            if needle.replace(" ", "") in compact:
                findings.setdefault(needle, (state, fix, 0))
                s, f, n = findings[needle]
                findings[needle] = (s, f, n + 1)
    return findings


legacy_wire = (workdir / "wire_legacy.log").read_text().splitlines()
extra = ['POST /mcp  Mcp-Session-Id: 1868a90c', '{"method": "logging/setLevel", "params": {"level": "info"}}',
         '{"method": "sampling/createMessage", "params": {"includeContext": "thisServer"}}']
for needle, (state, fix, count) in lint(legacy_wire + extra).items():
    print(f"[{state.upper():<10}] {needle:<34} x{count}  -> {fix}")

### 9. 已弃用的特性什么时候可以消失？
弃用窗口期（Deprecation Window）从弃用它的修订版<strong>发布</strong>时开始计算；最早移除时间是窗口期结束当天或之后作为当前版本发布的第一个修订版。下面 2026-07-28 之后的发布日期是为了练习而<strong>虚构</strong>的。

In [ ]:
from datetime import date


def add_months(d, months):
    y, m = divmod(d.month - 1 + months, 12)
    return date(d.year + y, m + 1, min(d.day, 28))


def earliest_removal(deprecated_in, releases, months=12, days=None):
    """First release on/after the window ends. `days` models an expedited (security) window, min 90."""
    if days is not None:
        assert days >= 90, "expedited removal still needs at least ninety days"
        window_end = date.fromordinal(deprecated_in.toordinal() + days)
    else:
        assert months >= 12, "the standard window is at least twelve months"
        window_end = add_months(deprecated_in, months)
    return window_end, next((r for r in sorted(releases) if r >= window_end), None)


hypothetical_releases = [date(2026, 12, 15), date(2027, 6, 30), date(2027, 11, 18), date(2028, 5, 2)]
deprecated = date(2026, 7, 28)   # Roots, Sampling, Logging, DCR
for label, kwargs in [("standard 12 months", {}), ("SEP asks for 18 months", {"months": 18}),
                      ("expedited, 120 days", {"days": 120})]:
    end, release = earliest_removal(deprecated, hypothetical_releases, **kwargs)
    print(f"{label:<24} window ends {end} -> earliest removal in revision released {release}")

注意 12 个月的情况落在了 2027-11-18，而不是 2027-07-28：移除只会<em>在某次发布中</em>发生，而 2027-06-30 那次发布来得太早了。

### 动手练习
1. 在第 3 节中，把双时代服务器换成<strong>只支持新版</strong>的服务器，并且只保留 `CLIENT_SUPPORTS = ["2027-03-01"]`。让 `call_with_negotiation` 报告一条有帮助的错误信息，同时引用双方的版本列表。
2. 在第 5 节中，修改 `legacy_server.py`，让它在 `initialize` 之前<strong>从不回应</strong>请求（删除 `-32600` 的回复）。运行 `Client(params, read_timeout_seconds=3)`，观察 `mode="auto"` 需要多长时间才回退。规范中的哪句话涵盖了这种情况？
3. 为第 8 节中的检查器再添加三条规则，针对那些<em>不是</em>方法名的变更：旧的资源未找到错误码 `-32002`、旧的 `UnsupportedProtocolVersion` 错误码 `-32004`，以及一个缺少 `resultType` 的 `tools/call` 结果。它们分别应当迁移成什么？
4. 利用兼容性矩阵，写一段话回答：“我们的客户端全都使用 2025-11-25 的 SDK。我们能部署一个只支持新版的服务器吗？”